In [2]:
# ============================================================
# 1. IMPORT LIBRARIES AND CONFIGURE THE TRANSFORMATION
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T

# Use UTC consistently for processing timestamps
spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

# Project configuration
BRONZE_PATH = "Files/bronze/synthea"
SILVER_SCHEMA = "silver"
SOURCE_SYSTEM = "Synthea CSV November 2021"

SOURCE_FILE = "payers.csv"
SOURCE_TABLE = "payers"
TARGET_TABLE = f"{SILVER_SCHEMA}.payers"

# Create the Silver schema if it does not exist
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}"
)

print("Configuration completed.")
print(f"Bronze source: {BRONZE_PATH}")
print(f"Source file: {SOURCE_FILE}")
print(f"Target table: {TARGET_TABLE}")

StatementMeta(, 1f2a9971-00dc-4a46-b506-6dcc25e2f1a7, 4, Finished, Available, Finished, False)

Configuration completed.
Bronze source: Files/bronze/synthea
Source file: payers.csv
Target table: silver.payers


In [3]:
# ============================================================
# 2. INGEST THE RAW BRONZE PAYERS CSV FILE
# ============================================================

payer_file_path = (
    f"{BRONZE_PATH}/{SOURCE_FILE}"
)

payers_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(payer_file_path)
)

raw_payer_count = (
    payers_raw.count()
)

print(
    f"Source table: {SOURCE_TABLE}"
)

print(
    f"Source file: {payer_file_path}"
)

print(
    f"Rows loaded: {raw_payer_count:,}"
)

print(
    f"Columns loaded: "
    f"{len(payers_raw.columns)}"
)

payers_raw.printSchema()

StatementMeta(, 1f2a9971-00dc-4a46-b506-6dcc25e2f1a7, 5, Finished, Available, Finished, False)

Source table: payers
Source file: Files/bronze/synthea/payers.csv
Rows loaded: 10
Columns loaded: 21
root
 |-- Id: string (nullable = true)
 |-- NAME: string (nullable = true)
 |-- ADDRESS: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE_HEADQUARTERED: string (nullable = true)
 |-- ZIP: string (nullable = true)
 |-- PHONE: string (nullable = true)
 |-- AMOUNT_COVERED: string (nullable = true)
 |-- AMOUNT_UNCOVERED: string (nullable = true)
 |-- REVENUE: string (nullable = true)
 |-- COVERED_ENCOUNTERS: string (nullable = true)
 |-- UNCOVERED_ENCOUNTERS: string (nullable = true)
 |-- COVERED_MEDICATIONS: string (nullable = true)
 |-- UNCOVERED_MEDICATIONS: string (nullable = true)
 |-- COVERED_PROCEDURES: string (nullable = true)
 |-- UNCOVERED_PROCEDURES: string (nullable = true)
 |-- COVERED_IMMUNIZATIONS: string (nullable = true)
 |-- UNCOVERED_IMMUNIZATIONS: string (nullable = true)
 |-- UNIQUE_CUSTOMERS: string (nullable = true)
 |-- QOLS_AVG: string (nullab

In [4]:
# ============================================================
# 3. INSPECT RAW PAYER RECORDS
# ============================================================

display(
    payers_raw
)

print(
    payers_raw.columns
)

StatementMeta(, 1f2a9971-00dc-4a46-b506-6dcc25e2f1a7, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 76cfae56-7884-4900-8fc5-8553527a2838)

['Id', 'NAME', 'ADDRESS', 'CITY', 'STATE_HEADQUARTERED', 'ZIP', 'PHONE', 'AMOUNT_COVERED', 'AMOUNT_UNCOVERED', 'REVENUE', 'COVERED_ENCOUNTERS', 'UNCOVERED_ENCOUNTERS', 'COVERED_MEDICATIONS', 'UNCOVERED_MEDICATIONS', 'COVERED_PROCEDURES', 'UNCOVERED_PROCEDURES', 'COVERED_IMMUNIZATIONS', 'UNCOVERED_IMMUNIZATIONS', 'UNIQUE_CUSTOMERS', 'QOLS_AVG', 'MEMBER_MONTHS']


In [5]:
# ============================================================
# 4. STANDARDIZE PAYER COLUMN NAMES
# ============================================================

payer_column_mapping = {
    "Id": "payer_id",
    "ID": "payer_id",
    "NAME": "payer_name",
    "ADDRESS": "address",
    "CITY": "city",
    "STATE_HEADQUARTERED": "state_headquartered",
    "ZIP": "zip",
    "PHONE": "phone",
    "AMOUNT_COVERED": "amount_covered",
    "AMOUNT_UNCOVERED": "amount_uncovered",
    "REVENUE": "revenue",
    "COVERED_ENCOUNTERS": "covered_encounters",
    "UNCOVERED_ENCOUNTERS": "uncovered_encounters",
    "COVERED_MEDICATIONS": "covered_medications",
    "UNCOVERED_MEDICATIONS": "uncovered_medications",
    "COVERED_PROCEDURES": "covered_procedures",
    "UNCOVERED_PROCEDURES": "uncovered_procedures",
    "COVERED_IMMUNIZATIONS": "covered_immunizations",
    "UNCOVERED_IMMUNIZATIONS": "uncovered_immunizations",
    "UNIQUE_CUSTOMERS": "unique_customers",
    "QOLS_AVG": "quality_of_life_score_average",
    "MEMBER_MONTHS": "member_months"
}


payers_standardized = payers_raw

for old_name in payers_raw.columns:

    if old_name in payer_column_mapping:

        new_name = payer_column_mapping[
            old_name
        ]

    else:

        new_name = (
            old_name
            .strip()
            .lower()
        )

    payers_standardized = (
        payers_standardized
        .withColumnRenamed(
            old_name,
            new_name
        )
    )


# Stop if the renaming process creates
# duplicate column names
if (
    len(payers_standardized.columns)
    !=
    len(set(payers_standardized.columns))
):

    raise ValueError(
        "Duplicate column names were created "
        "while standardizing payers."
    )


print(
    f"Payers: "
    f"{len(payers_raw.columns)} source columns -> "
    f"{len(payers_standardized.columns)} "
    f"standardized columns"
)

print(
    payers_standardized.columns
)

StatementMeta(, 1f2a9971-00dc-4a46-b506-6dcc25e2f1a7, 7, Finished, Available, Finished, False)

Payers: 21 source columns -> 21 standardized columns
['payer_id', 'payer_name', 'address', 'city', 'state_headquartered', 'zip', 'phone', 'amount_covered', 'amount_uncovered', 'revenue', 'covered_encounters', 'uncovered_encounters', 'covered_medications', 'uncovered_medications', 'covered_procedures', 'uncovered_procedures', 'covered_immunizations', 'uncovered_immunizations', 'unique_customers', 'quality_of_life_score_average', 'member_months']


In [6]:
# ============================================================
# 5. CLEAN PAYER STRING VALUES
# ============================================================


def clean_string_values(df):

    cleaned_df = df

    for column_name in df.columns:

        cleaned_df = (
            cleaned_df
            .withColumn(
                column_name,
                F.when(
                    F.trim(
                        F.col(column_name)
                    ) == "",
                    F.lit(None)
                )
                .otherwise(
                    F.trim(
                        F.col(column_name)
                    )
                )
            )
        )

    return cleaned_df


# Count payer records before cleaning
payer_count_before_cleaning = (
    payers_standardized.count()
)


# Remove surrounding spaces and convert
# empty strings to proper null values
payers_cleaned = clean_string_values(
    payers_standardized
)


# Count payer records after cleaning
payer_count_after_cleaning = (
    payers_cleaned.count()
)


print(
    f"Payers: "
    f"{payer_count_before_cleaning:,} rows "
    f"before cleaning, "
    f"{payer_count_after_cleaning:,} rows "
    f"after cleaning"
)


# Stop if cleaning unexpectedly changes
# the number of payer records
if (
    payer_count_before_cleaning
    !=
    payer_count_after_cleaning
):

    raise ValueError(
        "Payer row count changed during "
        "string-value cleaning."
    )


print(
    "Payer string cleaning and "
    "row-count reconciliation passed."
)

StatementMeta(, 1f2a9971-00dc-4a46-b506-6dcc25e2f1a7, 8, Finished, Available, Finished, False)

Payers: 10 rows before cleaning, 10 rows after cleaning
Payer string cleaning and row-count reconciliation passed.


In [7]:
# ============================================================
# 6. TRANSFORM PAYER DATA TYPES
# ============================================================

payers_silver = (
    payers_cleaned

    # Preserve identifier and contact fields as strings
    .withColumn(
        "payer_id",
        F.col("payer_id").cast(
            T.StringType()
        )
    )
    .withColumn(
        "zip",
        F.col("zip").cast(
            T.StringType()
        )
    )
    .withColumn(
        "phone",
        F.col("phone").cast(
            T.StringType()
        )
    )

    # Convert financial fields to
    # fixed-precision decimal values
    .withColumn(
        "amount_covered",
        F.col(
            "amount_covered"
        ).cast(
            T.DecimalType(18, 2)
        )
    )
    .withColumn(
        "amount_uncovered",
        F.col(
            "amount_uncovered"
        ).cast(
            T.DecimalType(18, 2)
        )
    )
    .withColumn(
        "revenue",
        F.col("revenue").cast(
            T.DecimalType(18, 2)
        )
    )

    # Convert utilization fields to whole numbers
    .withColumn(
        "covered_encounters",
        F.col(
            "covered_encounters"
        ).cast(
            T.LongType()
        )
    )
    .withColumn(
        "uncovered_encounters",
        F.col(
            "uncovered_encounters"
        ).cast(
            T.LongType()
        )
    )
    .withColumn(
        "covered_medications",
        F.col(
            "covered_medications"
        ).cast(
            T.LongType()
        )
    )
    .withColumn(
        "uncovered_medications",
        F.col(
            "uncovered_medications"
        ).cast(
            T.LongType()
        )
    )
    .withColumn(
        "covered_procedures",
        F.col(
            "covered_procedures"
        ).cast(
            T.LongType()
        )
    )
    .withColumn(
        "uncovered_procedures",
        F.col(
            "uncovered_procedures"
        ).cast(
            T.LongType()
        )
    )
    .withColumn(
        "covered_immunizations",
        F.col(
            "covered_immunizations"
        ).cast(
            T.LongType()
        )
    )
    .withColumn(
        "uncovered_immunizations",
        F.col(
            "uncovered_immunizations"
        ).cast(
            T.LongType()
        )
    )
    .withColumn(
        "unique_customers",
        F.col(
            "unique_customers"
        ).cast(
            T.LongType()
        )
    )
    .withColumn(
        "member_months",
        F.col(
            "member_months"
        ).cast(
            T.LongType()
        )
    )

    # Convert the average quality-of-life
    # score to a numeric measurement
    .withColumn(
        "quality_of_life_score_average",
        F.col(
            "quality_of_life_score_average"
        ).cast(
            T.DoubleType()
        )
    )
)


payers_silver.printSchema()

StatementMeta(, 1f2a9971-00dc-4a46-b506-6dcc25e2f1a7, 9, Finished, Available, Finished, False)

root
 |-- payer_id: string (nullable = true)
 |-- payer_name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state_headquartered: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- amount_covered: decimal(18,2) (nullable = true)
 |-- amount_uncovered: decimal(18,2) (nullable = true)
 |-- revenue: decimal(18,2) (nullable = true)
 |-- covered_encounters: long (nullable = true)
 |-- uncovered_encounters: long (nullable = true)
 |-- covered_medications: long (nullable = true)
 |-- uncovered_medications: long (nullable = true)
 |-- covered_procedures: long (nullable = true)
 |-- uncovered_procedures: long (nullable = true)
 |-- covered_immunizations: long (nullable = true)
 |-- uncovered_immunizations: long (nullable = true)
 |-- unique_customers: long (nullable = true)
 |-- quality_of_life_score_average: double (nullable = true)
 |-- member_months: long (nullable = true)



In [8]:
# ============================================================
# 7. VALIDATE PAYER DATA QUALITY
# ============================================================

payer_validation = (
    payers_silver
    .agg(
        # Record and primary-key checks
        F.count("*").alias(
            "row_count"
        ),

        F.countDistinct(
            "payer_id"
        ).alias(
            "distinct_payer_ids"
        ),

        F.sum(
            F.when(
                F.col("payer_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_payer_ids"
        ),

        F.sum(
            F.when(
                F.col("payer_name").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_payer_names"
        ),

        # Financial-value checks
        F.sum(
            F.when(
                F.col("amount_covered") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_amount_covered"
        ),

        F.sum(
            F.when(
                F.col("amount_uncovered") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_amount_uncovered"
        ),

        F.sum(
            F.when(
                F.col("revenue") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_revenue"
        ),

        # Utilization checks
        F.sum(
            F.when(
                F.col("covered_encounters") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_covered_encounters"
        ),

        F.sum(
            F.when(
                F.col("uncovered_encounters") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_uncovered_encounters"
        ),

        F.sum(
            F.when(
                F.col("covered_medications") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_covered_medications"
        ),

        F.sum(
            F.when(
                F.col("uncovered_medications") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_uncovered_medications"
        ),

        F.sum(
            F.when(
                F.col("covered_procedures") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_covered_procedures"
        ),

        F.sum(
            F.when(
                F.col("uncovered_procedures") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_uncovered_procedures"
        ),

        F.sum(
            F.when(
                F.col("unique_customers") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_unique_customers"
        ),

        F.sum(
            F.when(
                F.col("member_months") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_member_months"
        ),

        # Quality-of-life score check
        F.sum(
            F.when(
                (
                    F.col(
                        "quality_of_life_score_average"
                    ) < 0
                )
                |
                (
                    F.col(
                        "quality_of_life_score_average"
                    ) > 1
                ),
                1
            ).otherwise(0)
        ).alias(
            "qols_outside_expected_range"
        )
    )
)

display(
    payer_validation
)

StatementMeta(, 1f2a9971-00dc-4a46-b506-6dcc25e2f1a7, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fda4604e-3de4-40fc-b838-c7273cbc5b30)

In [9]:
# ============================================================
# 8.1 PROFILE PAYER FINANCIAL AND COVERAGE VALUES
# ============================================================

payer_profile = (
    payers_silver

    # Calculate the percentage of the total amount
    # represented by payer-covered amounts
    .withColumn(
        "financial_coverage_percentage",
        F.when(
            (
                F.col("amount_covered")
                +
                F.col("amount_uncovered")
            ) > 0,
            F.round(
                (
                    F.col("amount_covered")
                    /
                    (
                        F.col("amount_covered")
                        +
                        F.col("amount_uncovered")
                    )
                ) * 100,
                2
            )
        )
    )

    # Calculate the percentage of encounters
    # reported as covered
    .withColumn(
        "encounter_coverage_percentage",
        F.when(
            (
                F.col("covered_encounters")
                +
                F.col("uncovered_encounters")
            ) > 0,
            F.round(
                (
                    F.col("covered_encounters")
                    /
                    (
                        F.col("covered_encounters")
                        +
                        F.col("uncovered_encounters")
                    )
                ) * 100,
                2
            )
        )
    )

    .select(
        "payer_name",
        "amount_covered",
        "amount_uncovered",
        "financial_coverage_percentage",
        "covered_encounters",
        "uncovered_encounters",
        "encounter_coverage_percentage",
        "unique_customers",
        "member_months"
    )

    .orderBy(
        F.desc(
            "amount_covered"
        )
    )
)

display(
    payer_profile
)

StatementMeta(, 1f2a9971-00dc-4a46-b506-6dcc25e2f1a7, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1f3e502e-819e-4fce-b52f-f4fa2b0015ff)

In [10]:
# ============================================================
# 9. ADD RECORD HASH AND SILVER PROCESSING METADATA
# ============================================================

# Source-derived fields included in the payer hash
payer_hash_columns = [
    "payer_id",
    "payer_name",
    "address",
    "city",
    "state_headquartered",
    "zip",
    "phone",
    "amount_covered",
    "amount_uncovered",
    "revenue",
    "covered_encounters",
    "uncovered_encounters",
    "covered_medications",
    "uncovered_medications",
    "covered_procedures",
    "uncovered_procedures",
    "covered_immunizations",
    "uncovered_immunizations",
    "unique_customers",
    "quality_of_life_score_average",
    "member_months"
]


# Prepare each field for consistent hash generation
hash_input_columns = [
    F.coalesce(
        F.col(column_name).cast("string"),
        F.lit("<NULL>")
    )
    for column_name in payer_hash_columns
]


payers_silver = (
    payers_silver

    # Create a SHA-256 fingerprint for
    # each payer record
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *hash_input_columns
            ),
            256
        )
    )

    # Add source lineage information
    .withColumn(
        "source_system",
        F.lit(SOURCE_SYSTEM)
    )
    .withColumn(
        "source_table",
        F.lit(SOURCE_TABLE)
    )
    .withColumn(
        "source_file",
        F.lit(SOURCE_FILE)
    )

    # Record the Silver processing time
    .withColumn(
        "silver_processed_timestamp",
        F.current_timestamp()
    )
)


print(
    "Record hash and Silver processing "
    "metadata added to payers."
)


payers_silver.select(
    "payer_id",
    "payer_name",
    "record_hash",
    "source_system",
    "source_table",
    "source_file",
    "silver_processed_timestamp"
).show(
    10,
    truncate=False
)

StatementMeta(, 1f2a9971-00dc-4a46-b506-6dcc25e2f1a7, 12, Finished, Available, Finished, False)

Record hash and Silver processing metadata added to payers.
+------------------------------------+----------------------+----------------------------------------------------------------+-------------------------+------------+-----------+--------------------------+
|payer_id                            |payer_name            |record_hash                                                     |source_system            |source_table|source_file|silver_processed_timestamp|
+------------------------------------+----------------------+----------------------------------------------------------------+-------------------------+------------+-----------+--------------------------+
|b3221cfc-24fb-339e-823d-bc4136cbc4ed|Dual Eligible         |4048e91f52189f3fb2e674285e3255301901de0e76cb76bc98936bb067df6a4c|Synthea CSV November 2021|payers      |payers.csv |2026-10-21 04:22:54.570748|
|7caa7254-5050-3b5e-9eae-bd5ea30e809c|Medicare              |a31bb42c5116c111c71ab13ffcd98cbb445c8318ebab80534a3895da1b2

In [11]:
# ============================================================
# 10. WRITE THE PAYERS TABLE TO THE SILVER LAYER
# ============================================================

(
    payers_silver.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TARGET_TABLE
    )
)

print(
    f"Silver table written successfully: "
    f"{TARGET_TABLE}"
)

StatementMeta(, 1f2a9971-00dc-4a46-b506-6dcc25e2f1a7, 13, Finished, Available, Finished, False)

Silver table written successfully: silver.payers


In [12]:
# ============================================================
# 11. VERIFY THE SAVED SILVER PAYERS TABLE
# ============================================================

saved_silver_payers = (
    spark.table(
        TARGET_TABLE
    )
)


saved_payer_validation = (
    saved_silver_payers
    .agg(
        F.count("*").alias(
            "saved_row_count"
        ),

        F.countDistinct(
            "payer_id"
        ).alias(
            "distinct_payer_ids"
        ),

        F.sum(
            F.when(
                F.col("payer_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_payer_ids"
        ),

        F.sum(
            F.when(
                F.col("record_hash").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_record_hashes"
        ),

        F.sum(
            F.when(
                F.col(
                    "silver_processed_timestamp"
                ).isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_processing_timestamps"
        )
    )
)


validation_result = (
    saved_payer_validation.first()
)


saved_payer_count = (
    validation_result[
        "saved_row_count"
    ]
)

saved_distinct_payer_count = (
    validation_result[
        "distinct_payer_ids"
    ]
)

null_saved_payer_ids = (
    validation_result[
        "null_payer_ids"
    ]
)

null_saved_record_hashes = (
    validation_result[
        "null_record_hashes"
    ]
)

null_saved_processing_timestamps = (
    validation_result[
        "null_processing_timestamps"
    ]
)


print(
    f"Silver payers row count: "
    f"{saved_payer_count:,}"
)

print(
    f"Distinct payer IDs: "
    f"{saved_distinct_payer_count:,}"
)

print(
    f"Null payer IDs: "
    f"{null_saved_payer_ids:,}"
)

print(
    f"Null record hashes: "
    f"{null_saved_record_hashes:,}"
)

print(
    f"Null processing timestamps: "
    f"{null_saved_processing_timestamps:,}"
)


# Enforce final saved-table checks
if saved_payer_count != 10:

    raise ValueError(
        "Payer row-count reconciliation failed. "
        f"Expected 10 but found "
        f"{saved_payer_count:,}."
    )


if saved_distinct_payer_count != 10:

    raise ValueError(
        "Payer ID uniqueness validation failed."
    )


if null_saved_payer_ids != 0:

    raise ValueError(
        "Null payer IDs were found "
        "in the saved Silver table."
    )


if null_saved_record_hashes != 0:

    raise ValueError(
        "Null payer record hashes were found "
        "in the saved Silver table."
    )


if null_saved_processing_timestamps != 0:

    raise ValueError(
        "Null processing timestamps were found "
        "in the saved Silver table."
    )


print(
    "Silver payer-table verification passed."
)


saved_silver_payers.printSchema()

display(
    saved_silver_payers
    .orderBy("payer_name")
)


StatementMeta(, 1f2a9971-00dc-4a46-b506-6dcc25e2f1a7, 14, Finished, Available, Finished, False)

Silver payers row count: 10
Distinct payer IDs: 10
Null payer IDs: 0
Null record hashes: 0
Null processing timestamps: 0
Silver payer-table verification passed.
root
 |-- payer_id: string (nullable = true)
 |-- payer_name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state_headquartered: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- amount_covered: decimal(18,2) (nullable = true)
 |-- amount_uncovered: decimal(18,2) (nullable = true)
 |-- revenue: decimal(18,2) (nullable = true)
 |-- covered_encounters: long (nullable = true)
 |-- uncovered_encounters: long (nullable = true)
 |-- covered_medications: long (nullable = true)
 |-- uncovered_medications: long (nullable = true)
 |-- covered_procedures: long (nullable = true)
 |-- uncovered_procedures: long (nullable = true)
 |-- covered_immunizations: long (nullable = true)
 |-- uncovered_immunizations: long (nullable = true)


SynapseWidget(Synapse.DataFrame, cc7e883e-c87c-4611-912d-20c09d8ab0c0)